# Exercise 5: Use Flux for Logistic Regression

* Get the Churn_Modelling.csv data that was used to demonstrate the GLM package. The data is available at  https://github.com/selva86/datasets/tree/master
* Follow the steps described in the Flux Demos and in the GLM demos shown during the sessions
* Evaluate the accuracy of the model’s prediction, using a confusion matrix and a ROC curve.
* Submit your project following the submission guidelines

In [ ]:
import Pkg; 
# Pkg.add("MLBase");
# Pkg.add("MLDatasets")

using Flux, MLDatasets, DataFrames,CSV,Random,Plots
using Flux, MLDatasets, DataFrames
using Flux: logitcrossentropy, normalise, onecold, onehotbatch,crossentropy
using Statistics: mean
using RDatasets
using MLBase
using PrettyTables

Pkg.offline(true)

In [ ]:

findaccuracy(predictedvals,groundtruthvals) = sum(predictedvals.==groundtruthvals)/length(groundtruthvals)


calcular_tpr_fpr (generic function with 2 methods)

In [ ]:
data = CSV.File("./dat/Churn_Modelling.csv")
df = DataFrame(data)


### Tratamiento de información

In [ ]:
df.Gender .= ifelse.(df.Gender .== "Male", 1, 0)
df.Exited .= ifelse.(df.Exited .== 1, 1, 0)
numeric_columns = names(df)[map(x -> eltype(df[!, x]) <: Number, names(df))]
df_numeric = df[:, numeric_columns]

### Obteniendo matrices X y y

In [ ]:
X = Matrix(df_numeric[:, Not(:Exited)]) # Convert DataFrame to numeric array (X)
X .= (X .- mean(X, dims=1)) ./ std(X, dims=1)
y = df[:, :Exited]

### Dividiendo información 70% para training y 20% para test

In [ ]:
train_size = Int(0.7 * size(X, 1))
println(train_size)
onehot_labels = onehotbatch(y, [0,1])'

In [54]:
X_train, X_test = X[1:train_size, :], X[train_size+1:end, :]
# y_train, y_test = y[1:train_size, :], y[train_size+1:end, :]
y_train, y_test = onehot_labels[1:train_size, :], onehot_labels[train_size+1:end, :]

@show size(X_train)
@show size(X_test)
@show size(y_train)
@show size(y_test)

size(X_train) = (7000, 11)
size(X_test) = (3000, 11)
size(y_train) = (7000, 2)
size(y_test) = (3000, 2)


(3000, 2)

### Enternamiento

In [ ]:
# model = Chain(Dense(size(X, 2), 1, σ),)
# @show size(X, 2)
model = Chain(Dense(size(X, 2), 2))

loss(m,x, y) = logitcrossentropy(m(x), y)
# loss(m,x, y) = crossentropy(m(x), y)
opt = Descent(0.1)
for i in 1:100
    Flux.train!(loss, model, [(X_train', y_train')], opt)
end


size(X, 2) = 11


### Generando predicción

In [63]:
y_pred = model(X_test') .> 0.5  # Threshold of 0.5 to convert to 0 or 1
@show size(y_test)
@show size(y_pred)


size(y_test) = (3000, 2)
size(y_pred) = (2, 3000)


(2, 3000)

### Obteniendo matriz de confusion

In [64]:

conf_matrix = zeros(Int, 2, 2) # Confusion matrix
for i in 1:length(y_pred)
    conf_matrix[Int(y_test[i]) + 1, Int(y_pred[i]) + 1] += 1
end

println("Matriz de confusión:")
pretty_table(conf_matrix)


Matriz de confusión:
┌────────┬────────┐
│ Col. 1 │ Col. 2 │
├────────┼────────┤
│   1742 │   1258 │
│   1799 │   1201 │
└────────┴────────┘


### Calculando accuracy

In [65]:
accuracy_score = findaccuracy(y_test', y_pred)
println("\nAccuracy: $accuracy_score")



Accuracy: 0.6985
